# English → Japanese Translation Error Annotation Pipeline

This notebook builds an error-annotated dataset of English→Japanese translations in three steps:

| Step | Description | Environment |
|------|-------------|-------------|
| **1** | Sample 500 sentence pairs from JParaCrawl | Local |
| **2** | Generate candidate translations with **Qwen2.5-7B-Instruct** (4-bit NF4) | Google Colab (free T4 GPU) |
| **3** | Annotate translation errors with **Gemini** (`gemini-3.1-flash-lite-preview`) via the Gemini API, processing 500 entries in batches of 10 | Google Colab (free T4 GPU) |

An LLM-as-judge (**Gemini `gemini-3.1-flash-lite-preview`**) scores each candidate for five error
categories (Terminology, Accuracy, Fluency/Grammar, Style/Register, Locale/Formatting) and assigns
a severity (major / minor / none).

**Inputs** (in `kb/`):
- `kb/eng-jap.tsv` — JParaCrawl sentence pairs

**Intermediate** (in `kb/`):
- `kb/candidates_for_annotation.jsonl` — Qwen2.5-7B-Instruct candidate translations

**Outputs** (in `kb/`):
- `kb/annotations_raw.jsonl` — full Gemini annotation results
- `kb/splits/error_id_train.jsonl` (300 rows) and `kb/splits/error_id_test.jsonl` (200 rows)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Imports & Setup

In [ ]:
%pip install python-dotenv tqdm pandas --quiet

import pandas as pd
import json
import random
from tqdm.notebook import tqdm
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 35.2 MB/s eta 0:00:00


## 2. Load JParaCrawl Sample *(Local)*

Read the full JParaCrawl TSV and sample 500 sentence pairs for annotation.
This step was run locally.

In [ ]:
JPARACRAWL_PATH = next(
    (p for p in [Path("/content/drive/MyDrive/GenAI Own/eng-jap.tsv"), Path("kb/eng-jap.tsv"), Path("../kb/eng-jap.tsv")] if p.exists()),
    None,
)
if JPARACRAWL_PATH is None:
    raise FileNotFoundError("Could not find eng-jap.tsv in any of the specified paths.")

print(f"Loading JParaCrawl from: {JPARACRAWL_PATH.resolve()}")

df_jpc = pd.read_csv(
    JPARACRAWL_PATH,
    sep="\t",
    header=None,
    names=["eng_id", "source_en", "jp_id", "reference_ja"],
    dtype=str,
).fillna("")

df_jpc["id"] = df_jpc["eng_id"] + "_" + df_jpc["jp_id"]

SAMPLE_SIZE = 500
df_sample = df_jpc.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Sample shape: {df_sample.shape}")
df_sample[["id", "source_en", "reference_ja"]].head(3)

Loading JParaCrawl from: /content/drive/MyDrive/GenAI Own/eng-jap.tsv
Sample shape: (500, 5)


,id,source_en,reference_ja
0,41423_204181,Don't keep company with such a bad boy.,そんな悪い子と友達になるな。
1,9354007_9678940,It wasn't our fault. Believe me.,私達のせいじゃないわ。信じてよ。
2,71054_149509,May I ask you a question?,質問をしてもいいですか。


## 2b. Install Translation Dependencies *(Google Colab — T4 GPU)*

> **Steps 2b–2g** were run on **Google Colab** with a free **T4 GPU** for faster inference.

Install `transformers`, `accelerate`, and `bitsandbytes` for 4-bit quantized Qwen2.5 inference.

In [ ]:
%pip install -q torch transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00


## 2c. Load Qwen2.5-7B-Instruct

Load the model with 4-bit NF4 quantization (`BitsAndBytesConfig`). Tokenizer is set to
`padding_side="left"` for correct batch-generation behaviour. Seeds are fixed for reproducibility.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

gpu_name = torch.cuda.get_device_name(0)
vram_allocated = torch.cuda.memory_allocated(0) / 1024**3
vram_reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"GPU: {gpu_name}")
print(f"VRAM allocated: {vram_allocated:.2f} GB")
print(f"VRAM reserved:  {vram_reserved:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

GPU: Tesla T4
VRAM allocated: 5.18 GB
VRAM reserved:  5.58 GB


## 2d. Translation Function

Define `translate_batch()` which applies Qwen's chat template with a translation system
prompt, generates with greedy decoding (`do_sample=False`), and returns only the newly
generated tokens. A sanity check translates the first sentence in `df_sample`.

In [ ]:
SYSTEM_MESSAGE = "You are a professional English-to-Japanese translator."

USER_TEMPLATE = (
    "Translate the following English text into Japanese. "
    "Output only the Japanese translation, nothing else.\n\n"
    "English: {source}"
)


def translate_batch(sources: list[str]) -> list[str]:
    conversations = []
    for src in sources:
        conversations.append([
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": USER_TEMPLATE.format(source=src)},
        ])

    templated = tokenizer.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer(
        templated,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=512,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = output_ids[:, prompt_len:]
    translations = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    return [t.strip() for t in translations]


first_row = df_sample.iloc[0]
sanity = translate_batch([first_row["source_en"]])[0]
print(f"EN:        {first_row['source_en']}")
print(f"Candidate: {sanity}")
print(f"Reference: {first_row['reference_ja']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


EN:        Don't keep company with such a bad boy.
Candidate: そんな不良と付き合わないでください。
Reference: そんな悪い子と友達になるな。


## 2e. Batch Inference

Translate all 500 sentences in batches of 4. Each result is written as a JSON line to
`candidates_for_annotation.jsonl` with annotation placeholder fields. The loop supports
resuming from a partial run by skipping already-completed IDs.

In [ ]:
import time

KB_DIR = JPARACRAWL_PATH.parent # Set KB_DIR to the parent directory of JPARACRAWL_PATH
OUTPUT_PATH = KB_DIR / "candidates_for_annotation.jsonl"
BATCH_SIZE = 4

# Ensure the directory exists before writing the file
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

completed_ids: set[str] = set()
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                completed_ids.add(json.loads(line)["id"])
    print(f"Resuming: {len(completed_ids)} rows already completed, skipping those.")

df_todo = df_sample[~df_sample["id"].isin(completed_ids)].reset_index(drop=True)
total_todo = len(df_todo)
print(f"Rows to translate: {total_todo}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

t_start = time.time()
done = 0

with open(OUTPUT_PATH, "a", encoding="utf-8") as fout:
    for batch_start in range(0, total_todo, BATCH_SIZE):
        batch = df_todo.iloc[batch_start : batch_start + BATCH_SIZE]
        sources = batch["source_en"].tolist()

        translations = translate_batch(sources)

        for (_, row), candidate_ja in zip(batch.iterrows(), translations):
            record = {
                "id": row["id"],
                "source_en": row["source_en"],
                "reference_ja": row["reference_ja"],
                "candidate_ja": candidate_ja,
                "has_error": None,
                "severity": None,
                "categories": [],
                "rationale": "",
            }
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

        fout.flush()
        done += len(batch)

        elapsed = time.time() - t_start
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total_todo - done) / rate if rate > 0 else 0
        print(
            f"\r[{done}/{total_todo}] "
            f"{done / total_todo * 100:.1f}% | "
            f"{rate:.1f} examples/sec | "
            f"ETA: {eta:.0f}s",
            end="",
        )

elapsed_total = time.time() - t_start
print(f"\nFinished {done} translations in {elapsed_total:.1f}s")
print(f"Output: {OUTPUT_PATH.resolve()}")

Rows to translate: 500
[500/500] 100.0% | 1.0 examples/sec | ETA: 0s
Finished 500 translations in 491.3s
Output: /content/drive/MyDrive/GenAI Own/candidates_for_annotation.jsonl


## 2f. Verify Output

Read back the generated JSONL and display a random sample to confirm translations look reasonable.

In [ ]:
df_candidates_out = pd.read_json(OUTPUT_PATH, lines=True, dtype={"id": str})
print(f"Total records: {len(df_candidates_out)}")

sample_rows = df_candidates_out.sample(n=5, random_state=RANDOM_SEED)
display(sample_rows[["source_en", "reference_ja", "candidate_ja"]])

Total records: 500


,source_en,reference_ja,candidate_ja
361,Tom doesn't know that I hate him.,トムは私がトムを嫌ってることを知りません,トムは私に恨まれていることを知りません。
73,This bird can imitate the human voice.,この鳥は人の声を真似できるんだ。,この鳥は人間の声を模倣できます。
374,He writes to his mother every now and then.,彼は時折母親に手紙を書く。,彼は時々母に手紙を書きます。
155,Tom rushed to his room.,トムは自分の部屋にダッシュした。,トムは部屋へ急いで行きました。
104,They sought to punish him for his crime but he...,彼は罪を犯したので罰を加えようとしたが、彼は逃走した。,彼の罪を罰しようとしたが、彼は逃げた。


## 2g. Save Generation Config

Persist the model name, quantization settings, generation kwargs, and prompt templates
to `generation_config.json` for reproducibility.

In [ ]:
generation_config = {
    "model_name": MODEL_ID,
    "quantization": {
        "method": "bitsandbytes",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
    },
    "generation_kwargs": {
        "do_sample": False,
        "max_new_tokens": 512,
        "repetition_penalty": 1.0,
    },
    "prompt_templates": {
        "system_message": SYSTEM_MESSAGE,
        "user_template": USER_TEMPLATE,
    },
    "tokenizer": {
        "padding_side": "left",
        "max_length": 1024,
        "truncation": True,
    },
    "seed": 42,
    "batch_size": BATCH_SIZE,
    "num_samples": len(df_sample),
}

KB_DIR = JPARACRAWL_PATH.parent # Set KB_DIR to the parent directory of JPARACRAWL_PATH
CONFIG_PATH = KB_DIR / "generation_config.json"
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(generation_config, f, indent=2, ensure_ascii=False)

print(f"Saved generation config to {CONFIG_PATH.resolve()}")
print(json.dumps(generation_config, indent=2))

Saved generation config to /content/drive/MyDrive/GenAI Own/generation_config.json
{
  "model_name": "Qwen/Qwen2.5-7B-Instruct",
  "quantization": {
    "method": "bitsandbytes",
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_use_double_quant": true
  },
  "generation_kwargs": {
    "do_sample": false,
    "max_new_tokens": 512,
    "repetition_penalty": 1.0
  },
  "prompt_templates": {
    "system_message": "You are a professional English-to-Japanese translator.",
    "user_template": "Translate the following English text into Japanese. Output only the Japanese translation, nothing else.\n\nEnglish: {source}"
  },
  "tokenizer": {
    "padding_side": "left",
    "max_length": 1024,
    "truncation": true
  },
  "seed": 42,
  "batch_size": 4,
  "num_samples": 500
}


## 3. Load Candidate Translations *(Google Colab)*

> **Steps 3–5** were run on **Google Colab**, using the **Gemini API** (`gemini-3.1-flash-lite-preview`) to annotate all 500 error entries batch by batch.

Load the JSONL file containing Qwen2.5-7B-Instruct candidate translations and merge
with the JParaCrawl sample on `[id, source_en]`.

In [ ]:
CANDIDATES_PATH = JPARACRAWL_PATH.parent / "candidates_for_annotation.jsonl" # Use the parent directory for candidate path

if not CANDIDATES_PATH.exists(): # Check if the path exists
    raise FileNotFoundError(
        f"Could not find candidates.jsonl at {CANDIDATES_PATH.resolve()}"
    )

print(f"Loading candidates from: {CANDIDATES_PATH.resolve()}")

df_candidates = pd.read_json(CANDIDATES_PATH, lines=True, dtype={"id": str})
print(f"Candidates shape: {df_candidates.shape}")

df_merged = df_sample[["id", "source_en", "reference_ja"]].merge(
    df_candidates[["id", "source_en", "candidate_ja"]],
    on=["id", "source_en"],
    how="inner",
)

assert df_merged.isna().sum().sum() == 0, "Merge introduced null values!"
print(f"Merged shape: {df_merged.shape}")
df_merged.head(3)

Loading candidates from: /content/drive/MyDrive/GenAI Own/candidates_for_annotation.jsonl
Candidates shape: (500, 8)
Merged shape: (500, 4)


,id,source_en,reference_ja,candidate_ja
0,41423_204181,Don't keep company with such a bad boy.,そんな悪い子と友達になるな。,そんな不良と付き合わないでください。
1,9354007_9678940,It wasn't our fault. Believe me.,私達のせいじゃないわ。信じてよ。,それは私たちの faultではありません。信じてください。
2,71054_149509,May I ask you a question?,質問をしてもいいですか。,質問させていただけますか？


## 4. Gemini API Setup & Batch Annotation

Configure the **Gemini API** (`gemini-3.1-flash-lite-preview`) with a structured system prompt
and annotate all 500 candidate translations in batches of 10, with a 60-second cooldown
between batches to stay within API rate limits.

In [3]:
import google.generativeai as genai
import json
import time
import os
from google.colab import userdata

# 1. Bypass Colab's internal proxy
os.environ['NO_PROXY'] = 'generativelanguage.googleapis.com'

# 2. Configure the API
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY, transport="rest")
except userdata.SecretNotFoundError:
    print("Error: Please set the 'GEMINI_API_KEY' secret in the Colab sidebar.")

# 3. Setup the Model
SYSTEM_INSTRUCTION = """
You are a professional English→Japanese translation quality evaluator.
Given an English source sentence, a candidate Japanese translation, and a reference Japanese translation, evaluate the candidate for errors.

### Error categories
1. Terminology – incorrect or inconsistent term usage
2. Accuracy – meaning additions, omissions, or distortions
3. Fluency/Grammar – unnatural phrasing, grammatical mistakes
4. Style/Register – inappropriate formality level or tone
5. Locale/Formatting – wrong number formats, date formats, punctuation conventions

### Severity
- major – the error changes meaning or seriously hinders comprehension
- minor – the error is noticeable but does not change core meaning
- none – no meaningful error detected

### Rules
- Do NOT penalise stylistic differences if the core meaning is preserved correctly.
- If `has_error` is false, `severity` MUST be "none" and `categories` MUST be [].
- Return ONLY valid JSON matching this schema:
{
  "has_error": true | false,
  "severity": "major" | "minor" | "none",
  "categories": ["Category Name"],
  "rationale": "string"
}
"""

model = genai.GenerativeModel(
    model_name='gemini-3.1-flash-lite-preview',
    system_instruction=SYSTEM_INSTRUCTION,
    generation_config={"response_mime_type": "application/json"}
)

# 4. The Batch Processing Function
def process_in_batches(input_file, output_file, batch_size=10):
    print(f"Starting batch annotation. Reading from {input_file}...")

    # Read all lines from the file
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    total_entries = len(lines)
    print(f"Total entries to process: {total_entries}")

    # Figure out where to resume if the script was interrupted
    processed_count = 0
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_count = sum(1 for line in f)
        print(f"Found existing output file. Resuming from entry {processed_count}...")

    # Process the remaining lines in chunks
    for i in range(processed_count, total_entries, batch_size):
        chunk = lines[i : i + batch_size]
        print(f"\n--- Processing Batch {i//batch_size + 1} (Entries {i+1} to {i + len(chunk)}) ---")

        # Open in "append" mode ('a') so we don't overwrite previous batches
        with open(output_file, 'a', encoding='utf-8') as outfile:
            for line in chunk:
                data = json.loads(line)
                prompt = (
                    f"Source (EN): {data.get('source_en')}\n"
                    f"Candidate (JA): {data.get('candidate_ja')}\n"
                    f"Reference (JA): {data.get('reference_ja')}"
                )

                # Retry loop for individual timeouts
                retries = 3
                for attempt in range(retries):
                    try:
                        response = model.generate_content(prompt)
                        evaluation = json.loads(response.text)

                        data['has_error'] = evaluation.get('has_error')
                        data['severity'] = evaluation.get('severity')
                        data['categories'] = evaluation.get('categories', [])
                        data['rationale'] = evaluation.get('rationale', '')

                        outfile.write(json.dumps(data, ensure_ascii=False) + '\n')
                        print(f"  ✓ Processed ID: {data.get('id', 'Unknown')}")
                        break # Break out of retry loop on success

                    except Exception as e:
                        if attempt < retries - 1:
                            print(f"  ! Error on ID {data.get('id')}. Retrying... ({e})")
                            time.sleep(5) # Brief pause before retry
                        else:
                            print(f"  X Failed to process ID {data.get('id')}. Skipping. Error: {e}")

        # After completing a batch, check if there are more items to process
        if i + batch_size < total_entries:
            print("Batch complete. Sleeping for 60 seconds to reset API rate limit...")
            time.sleep(60)

    print("\nAll entries processed successfully!")

# 5. Run the Script
INPUT_PATH = '/content/drive/MyDrive/GenAI Own/candidates_for_annotation.jsonl'
OUTPUT_PATH = '/content/drive/MyDrive/GenAI Own/gemini_annotated_results.jsonl'

process_in_batches(INPUT_PATH, OUTPUT_PATH, batch_size=10)

Starting batch annotation. Reading from /content/drive/MyDrive/GenAI Own/candidates_for_annotation.jsonl...
Total entries to process: 500
Found existing output file. Resuming from entry 0...

--- Processing Batch 1 (Entries 1 to 10) ---
  ✓ Processed ID: 41423_204181
  ✓ Processed ID: 9354007_9678940
  ✓ Processed ID: 71054_149509
  ✓ Processed ID: 264505_150052
  ✓ Processed ID: 9193501_11563342
  ✓ Processed ID: 283323_120685
  ✓ Processed ID: 35902_198709
  ✓ Processed ID: 43729_206481
  ✓ Processed ID: 279489_124505
  ✓ Processed ID: 12120017_3120246
Batch complete. Sleeping for 60 seconds to reset API rate limit...

--- Processing Batch 2 (Entries 11 to 20) ---
  ✓ Processed ID: 298089_105599
  ✓ Processed ID: 61271_223936
  ✓ Processed ID: 65550_228196
  ✓ Processed ID: 5747794_5748453
  ✓ Processed ID: 1500305_137771
  ✓ Processed ID: 293864_1383978


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 10536.94ms


  ✓ Processed ID: 6574071_6089608


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5508.16ms


  ✓ Processed ID: 73350_235969
  ✓ Processed ID: 314361_89350
  ✓ Processed ID: 1485672_201080
Batch complete. Sleeping for 60 seconds to reset API rate limit...

--- Processing Batch 3 (Entries 21 to 30) ---
  ✓ Processed ID: 275311_137924
  ✓ Processed ID: 239900_174569
  ✓ Processed ID: 263362_151194
  ✓ Processed ID: 297207_106480
  ✓ Processed ID: 49396_212117
  ✓ Processed ID: 256155_158375
  ✓ Processed ID: 39574_202363
  ✓ Processed ID: 10465226_231952
  ✓ Processed ID: 49114_211837
  ✓ Processed ID: 10937149_10937150
Batch complete. Sleeping for 60 seconds to reset API rate limit...

--- Processing Batch 4 (Entries 31 to 40) ---
  ✓ Processed ID: 36021_198826
  ✓ Processed ID: 53889_216584
  ✓ Processed ID: 269890_144674
  ✓ Processed ID: 502362_502354
  ✓ Processed ID: 249611_164896
  ✓ Processed ID: 1108772_4713
  ✓ Processed ID: 58672_221348
  ✓ Processed ID: 38407_201200
  ✓ Processed ID: 908915_186898
  ✓ Processed ID: 57656_220336
Batch complete. Sleeping for 60 seconds 

...

  ! Error on ID 7240911_10746979. Retrying... (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 35.868431059s.)


  ! Error on ID 7240911_10746979. Retrying... (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 30.5772639s.)


  X Failed to process ID 7240911_10746979. Skipping. Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 25.26985882s.


  ! Error on ID 36018_198822. Retrying... (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 24.966724382s.)


  ! Error on ID 36018_198822. Retrying... (429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 19.685941056s.)
  ✓ Processed ID: 36018_198822

All entries processed successfully!


In [ ]:
ANNOTATION_PROMPT = """\
You are a professional English→Japanese translation quality evaluator.

Given an English source sentence, a candidate Japanese translation, and a reference
Japanese translation, evaluate the candidate for errors.

### Error categories
1. **Terminology** – incorrect or inconsistent term usage
2. **Accuracy** – meaning additions, omissions, or distortions
3. **Fluency/Grammar** – unnatural phrasing, grammatical mistakes
4. **Style/Register** – inappropriate formality level or tone
5. **Locale/Formatting** – wrong number formats, date formats, punctuation conventions

### Severity
- **major** – the error changes meaning or seriously hinders comprehension
- **minor** – the error is noticeable but does not change core meaning
- **none** – no meaningful error detected

### Rules
- Do NOT penalise stylistic differences if the core meaning is preserved correctly.
- If `has_error` is false, `severity` MUST be "none" and `categories` MUST be [].
- Return ONLY valid JSON — no markdown fences, no explanation outside the JSON.

### Input
- **Source (EN):** {source_en}
- **Candidate (JA):** {candidate_ja}
- **Reference (JA):** {reference_ja}

### Required JSON output schema
{{
  "has_error": true | false,
  "severity": "major" | "minor" | "none",
  "categories": [],
  "rationale": "string"
}}
"""

print("Prompt template loaded. Placeholders: {source_en}, {candidate_ja}, {reference_ja}")

Prompt template loaded. Placeholders: {source_en}, {candidate_ja}, {reference_ja}


## 5. Verify Annotation Results

Load the Gemini-annotated JSONL and display the first few rows to confirm the annotations look correct.

In [4]:
import pandas as pd
from pathlib import Path

# Assuming OUTPUT_PATH is defined elsewhere, but re-defining for standalone execution if needed
# This path is based on previous cell outputs where `gemini_annotated_results.jsonl` is created.
OUTPUT_PATH = Path('/content/drive/MyDrive/GenAI Own/gemini_annotated_results.jsonl')

if not OUTPUT_PATH.exists():
    print(f"Error: The file {OUTPUT_PATH.name} does not exist at {OUTPUT_PATH.parent}.")
else:
    df_annotated_results = pd.read_json(OUTPUT_PATH, lines=True, dtype={'id': str})
    print(f"Successfully loaded {len(df_annotated_results)} annotations.")
    print("First 5 annotated results:")
    display(df_annotated_results.head(5))

Successfully loaded 499 annotations.
First 5 annotated results:


,id,source_en,reference_ja,candidate_ja,has_error,severity,categories,rationale
0,41423_204181,Don't keep company with such a bad boy.,そんな悪い子と友達になるな。,そんな不良と付き合わないでください。,False,none,[],The candidate translation accurately reflects ...
1,9354007_9678940,It wasn't our fault. Believe me.,私達のせいじゃないわ。信じてよ。,それは私たちの faultではありません。信じてください。,True,minor,[Terminology],The candidate leaves the English word 'fault' ...
2,71054_149509,May I ask you a question?,質問をしてもいいですか。,質問させていただけますか？,False,none,[],The candidate translation is grammatically cor...
3,264505_150052,"In his autobiography, he repeatedly refers to ...",自伝の中で彼はくりかえし不幸な少年時代に言及している。,彼の自伝では、彼は再三、不快な学校生活について言及しています。,True,minor,[Terminology],The term 'unhappy school days' is translated a...
4,9193501_11563342,The apple cake's ran out.,アップルケーキがなくなっちゃった。,APPLE ケーキがなくなりました。,True,minor,[Style/Register],Using half-width Latin characters for 'APPLE' ...
